# File 04 — Feature Engineering
**EHSL-ML+ | Disertasi Nanan Soekarna**

File ini mengubah skor index (yang formatnya masih "per aktor" / long format) jadi **1 baris = 1 responden**, dengan ~50-70 kolom fitur — matriks yang siap dipakai ML di File 05.

Mengikuti proposal (§3.11.5, Tabel 3.27-3.32), ada 3 kelompok fitur turunan:
1. **Interaction Features** — perkalian antar-skor (`IF_ij = x_i × x_j`)
2. **Temporal Pattern Features** — bagaimana skor berubah antar-tahap KUHAP (`T = t1,t2,t3,t4`)
3. **Ethical Consistency Vector (ECV)** — seberapa selaras VfV/FCNC/ID untuk tiap aktor (`Consistency = 1-σ`)

Ditambah skor dasar (dari File 02) dan variabel demografi.

**Catatan jujur soal desain:** proposal mendeskripsikan `IF_VfV,Actor` dan `IF_NREI,Actor` secara konsep (`x × ActorCategory_encoded`), tapi tidak menentukan cara encoding kategori aktor yang nominal (Investigator/Prosecutor/dst.) jadi 1 angka tunggal per responden — apalagi 1 responden bisa punya banyak aktor sekaligus. Daripada memaksa nilai kategorikal jadi angka secara sembarangan (yang bisa menyesatkan model), saya representasikan 2 interaksi ini lewat **kolom pivot per-aktor** (`VfV_Investigator`, `NREI_Prosecutor`, dst.) — secara fungsional itu tetap menangkap "bagaimana skor berinteraksi dengan aktor", cuma bentuknya per-aktor bukan 1 angka gabungan.

In [1]:
import numpy as np
import pandas as pd

df_actor = pd.read_csv("data_index_actor.csv")
df_resp = pd.read_csv("data_index_respondent.csv")
df_demo = pd.read_csv("data_respondent_clean.csv")

actor_categories = ["Investigator", "Prosecutor", "Judge", "Advocate", "CorrectionalOfficer"]
constructs = ["VfV", "FCNC", "ID"]
stage_map = {  # tahap KUHAP -> aktor yang diukur (Tabel 3.29)
    "t1": ["Investigator"],
    "t2": ["Prosecutor"],
    "t3": ["Judge", "Advocate"],
    "t4": ["CorrectionalOfficer"],
}
print(f"data_actor: {len(df_actor)} baris | data_respondent: {len(df_resp)} baris")

data_actor: 896 baris | data_respondent: 199 baris


## A. Skor Dasar (dari File 02)

6 skor index keseluruhan per responden — ini fondasi feature matrix.

In [2]:
base = df_resp[["respondent_id","VfV_overall","FCNC_overall","ID_overall",
                "MAS_overall","ECI_overall","GR_norm","NREI_overall"]].copy()
base = base.rename(columns={"GR_norm": "GR_overall"})
print(f"Kolom: {base.shape[1]-1}")
base.head(3)

Kolom: 7


,respondent_id,VfV_overall,FCNC_overall,ID_overall,MAS_overall,ECI_overall,GR_overall,NREI_overall
0,R0001,37.50,46.5625,50.9375,42.875,48.750,50.0,46.65
1,R0002,43.75,41.2500,56.7500,48.950,49.000,45.0,48.18
2,R0003,32.00,32.0000,33.2500,32.500,32.625,20.0,30.05


## B. Actor-Specific Pivot (representasi IF_VfV,Actor & IF_NREI,Actor)

Dari `data_actor` (long format), kita "putar" jadi wide: 1 kolom per kombinasi skor × aktor. Kalau seorang responden tidak menemui aktor tertentu, nilainya otomatis kosong (NaN) — bukan 0, karena 0 akan salah diartikan sebagai "skor terendah", padahal maknanya "tidak berlaku".

In [3]:
piv_vfv = df_actor.pivot_table(index="respondent_id", columns="actor_category", values="VfV_norm")
piv_vfv.columns = [f"VfV_{c}" for c in piv_vfv.columns]
piv_nrei = df_actor.pivot_table(index="respondent_id", columns="actor_category", values="NREI_actor")
piv_nrei.columns = [f"NREI_{c}" for c in piv_nrei.columns]
actor_pivot = piv_vfv.join(piv_nrei).reset_index()
print(f"Kolom: {actor_pivot.shape[1]-1}")
actor_pivot.head(3)

Kolom: 10


,respondent_id,VfV_Advocate,VfV_CorrectionalOfficer,VfV_Investigator,VfV_Judge,VfV_Prosecutor,NREI_Advocate,NREI_CorrectionalOfficer,NREI_Investigator,NREI_Judge,NREI_Prosecutor
0,R0001,50.0,20.0,55.0,25.00,NaN,58.4,39.05,46.95,42.2,NaN
1,R0002,30.0,25.0,45.0,56.25,62.5,40.2,38.40,59.60,44.5,58.2
2,R0003,70.0,15.0,5.0,20.00,50.0,48.4,24.85,15.60,25.8,35.6


## C. Interaction Features (Tabel 3.28)

4 dari 6 interaksi utama dihitung langsung sebagai perkalian skor ternormalisasi (skala 0-1, sesuai formula proposal `x_i × x_j` dengan `x` dalam [0,1]):

- `IF_VfV,FCNC`, `IF_FCNC,ID`, `IF_VfV,ID` — perkalian antar skor keseluruhan
- `IF_ID,Stage` — perkalian ID dengan proksi "seberapa jauh proses hukum dijalani" (`n_aktor_ditemui / 5`, karena proposal tidak memberi skema encoding stage yang eksplisit)

(2 sisanya — `IF_VfV,Actor` dan `IF_NREI,Actor` — sudah direpresentasikan di bagian B.)

**Catatan penting — keempat kolom `IF_*` ini TIDAK boleh dipakai sebagai fitur model** ketika targetnya kategori NREI. Alasannya: ketiganya adalah perkalian skor agregat, sehingga skor aslinya bisa dihitung balik secara aljabar —

```
VfV = 100 × √( IF_VfV_FCNC × IF_VfV_ID / IF_FCNC_ID )
```

— dan `IF_ID_Stage` adalah `ID_overall` yang diskalakan. File 05 mengeluarkan keempatnya dari himpunan fitur. Kolom-kolomnya tetap dihitung dan disimpan di sini karena diminta eksplisit oleh proposal (Tabel 3.28) dan berguna untuk analisis deskriptif — yang dilarang hanya memakainya sebagai prediktor NREI.

**Pengganti yang bersih:** `n_actors_encountered` (jumlah aktor yang benar-benar ditemui) kini **tetap dipertahankan** sebagai kolom tersendiri. Ia membawa informasi "sejauh mana proses hukum dijalani" — bagian yang substantif dari `IF_ID_Stage` — tanpa ikut membawa skor ID. Versi sebelumnya menghitung kolom ini lalu langsung membuangnya.

In [4]:
interact = base[["respondent_id"]].copy()
interact["IF_VfV_FCNC"] = (base["VfV_overall"]/100) * (base["FCNC_overall"]/100)
interact["IF_FCNC_ID"] = (base["FCNC_overall"]/100) * (base["ID_overall"]/100)
interact["IF_VfV_ID"] = (base["VfV_overall"]/100) * (base["ID_overall"]/100)

n_actors = df_actor.groupby("respondent_id")["actor_category"].count().rename("n_actors_encountered")
interact = interact.merge(n_actors, on="respondent_id", how="left")
interact["IF_ID_Stage"] = (base["ID_overall"]/100) * (interact["n_actors_encountered"]/5)
# n_actors_encountered TIDAK lagi dibuang: ia adalah pengganti BERSIH untuk
# IF_ID_Stage (membawa info "sejauh mana proses hukum dijalani" tanpa skor ID).
print(f"Kolom: {interact.shape[1]-1}")
interact.head(3)

Kolom: 5


,respondent_id,IF_VfV_FCNC,IF_FCNC_ID,IF_VfV_ID,n_actors_encountered,IF_ID_Stage
0,R0001,0.174609,0.237178,0.191016,4,0.4075
1,R0002,0.180469,0.234094,0.248281,5,0.5675
2,R0003,0.102400,0.106400,0.106400,5,0.3325


## D. Temporal Pattern Features (Tabel 3.29-3.30)

Ingat tahapan KUHAP: `t1`=Investigator, `t2`=Prosecutor, `t3`=rata-rata(Judge,Advocate), `t4`=CorrectionalOfficer. Untuk tiap konstruk (VfV/FCNC/ID), dihitung:
- Skor tiap tahap (`T_t1`-`T_t4`)
- `Trend` = t4−t1 (positif = membaik dari penyidikan ke pembinaan)
- `MaxDrop` = selisih skor tertinggi-terendah antar tahap
- `Var` = variasi skor antar tahap (rendah = konsisten di semua tahap)

**Penting:** kalau responden cuma mengalami 1 tahap (atau kurang dari 2 tahap valid), `Trend`/`MaxDrop`/`Var` diisi **kosong (NaN)**, bukan dipaksa dihitung — sesuai instruksi proposal eksplisit soal ini.

In [5]:
temporal_rows = []
for resp_id, g in df_actor.groupby("respondent_id"):
    row = {"respondent_id": resp_id}
    for construct in constructs:
        col = f"{construct}_norm"
        stage_vals = {}
        for stage, actors_in_stage in stage_map.items():
            vals = g.loc[g["actor_category"].isin(actors_in_stage), col]
            stage_vals[stage] = vals.mean() if len(vals) > 0 else np.nan
        for stage in ["t1", "t2", "t3", "t4"]:
            row[f"T_{construct}_{stage}"] = stage_vals[stage]

        valid_stages = [v for v in stage_vals.values() if not pd.isna(v)]
        if not pd.isna(stage_vals["t1"]) and not pd.isna(stage_vals["t4"]):
            row[f"Trend_{construct}"] = stage_vals["t4"] - stage_vals["t1"]
        else:
            row[f"Trend_{construct}"] = np.nan
        if len(valid_stages) >= 2:
            row[f"MaxDrop_{construct}"] = max(valid_stages) - min(valid_stages)
            row[f"Var_{construct}"] = np.std(valid_stages, ddof=1)
        else:
            row[f"MaxDrop_{construct}"] = np.nan
            row[f"Var_{construct}"] = np.nan
    temporal_rows.append(row)

temporal_df = pd.DataFrame(temporal_rows)
print(f"Kolom: {temporal_df.shape[1]-1}")
print(f"Responden dgn Trend_VfV kosong (cuma alami <2 tahap): {temporal_df['Trend_VfV'].isna().sum()}")
temporal_df[["respondent_id","T_VfV_t1","T_VfV_t2","T_VfV_t3","T_VfV_t4","Trend_VfV","MaxDrop_VfV","Var_VfV"]].head(3)

Kolom: 21
Responden dgn Trend_VfV kosong (cuma alami <2 tahap): 1


,respondent_id,T_VfV_t1,T_VfV_t2,T_VfV_t3,T_VfV_t4,Trend_VfV,MaxDrop_VfV,Var_VfV
0,R0001,55.0,NaN,37.500,20.0,-35.0,35.0,17.500000
1,R0002,45.0,62.5,43.125,25.0,-20.0,37.5,15.329497
2,R0003,5.0,50.0,45.000,15.0,10.0,45.0,22.126530


## E. Ethical Consistency Vector — ECV (Tabel 3.31)

Untuk tiap aktor yang ditemui, hitung seberapa **selaras** VfV/FCNC/ID orang itu menilai aktor tsb.

`Consistency = 1 - σ(VfV, FCNC, ID)` — makin dekat ke 1, makin selaras (ketiganya dinilai di level yang mirip). Makin dekat ke 0, makin "pecah" (misal VfV tinggi tapi FCNC sangat rendah — perlakuan dianggap manusiawi tapi tidak jujur prosedurnya, pola yang berbeda makna dari sekadar dirata-rata).

In [6]:
ecv_rows = []
for resp_id, g in df_actor.groupby("respondent_id"):
    row = {"respondent_id": resp_id}
    for _, r in g.iterrows():
        actor = r["actor_category"]
        vfv, fcnc, idd = r["VfV_norm"], r["FCNC_norm"], r["ID_norm"]
        row[f"ECV_{actor}_VfV"] = vfv
        row[f"ECV_{actor}_FCNC"] = fcnc
        row[f"ECV_{actor}_ID"] = idd
        triple = [vfv, fcnc, idd]
        if all(not pd.isna(x) for x in triple):
            row[f"ECV_{actor}_Consistency"] = 1 - (np.std(triple, ddof=0) / 100)
        else:
            row[f"ECV_{actor}_Consistency"] = np.nan
    ecv_rows.append(row)

ecv_df = pd.DataFrame(ecv_rows)
print(f"Kolom: {ecv_df.shape[1]-1}  (harapan: 4 x 5 aktor = 20)")
ecv_df.filter(regex="Investigator").head(3)

Kolom: 20  (harapan: 4 x 5 aktor = 20)


,ECV_Investigator_VfV,ECV_Investigator_FCNC,ECV_Investigator_ID,ECV_Investigator_Consistency
0,55.0,40.0,43.75,0.936262
1,45.0,55.0,80.00,0.852804
2,5.0,25.0,15.00,0.918350


## F. Variabel Demografi/Legal

Variabel kategorikal (gender, pendidikan, jenis kasus, institusi, region) di-encode jadi angka. Untuk pendidikan dipakai **ordinal encoding** (SD=0 < SMP=1 < SMA=2 < D3/S1=3, karena memang ada urutan). Untuk yang tidak punya urutan alami (jenis kasus, institusi, region), dipakai label encoding sederhana — nanti kalau modelnya sensitif terhadap urutan angka (misal regresi linear), sebaiknya diganti one-hot encoding di File 05.

In [7]:
demo = df_demo[["respondent_id","age","gender","education","case_type",
                 "sentence_length_months","time_served_months","institution","region"]].copy()
edu_order = {"SD": 0, "SMP": 1, "SMA": 2, "D3/S1": 3}
demo["education_encoded"] = demo["education"].map(edu_order)
demo["gender_encoded"] = demo["gender"].map({"Laki-laki": 0, "Perempuan": 1})
demo["case_type_encoded"] = demo["case_type"].astype("category").cat.codes
demo["institution_encoded"] = demo["institution"].astype("category").cat.codes
demo["region_encoded"] = demo["region"].astype("category").cat.codes
demo_final = demo[["respondent_id","age","gender_encoded","education_encoded","case_type_encoded",
                    "sentence_length_months","time_served_months","institution_encoded","region_encoded"]]
print(f"Kolom: {demo_final.shape[1]-1}")
demo_final.head(3)

Kolom: 8


,respondent_id,age,gender_encoded,education_encoded,case_type_encoded,sentence_length_months,time_served_months,institution_encoded,region_encoded
0,R0001,36.0,0,0,1,72,51,2,3
1,R0002,34.0,0,0,0,22,2,2,3
2,R0003,37.0,0,3,0,21,2,2,3


## Gabungkan Semua → Feature Matrix Final

In [8]:
feature_matrix = base.merge(actor_pivot, on="respondent_id", how="left") \
                      .merge(interact, on="respondent_id", how="left") \
                      .merge(temporal_df, on="respondent_id", how="left") \
                      .merge(ecv_df, on="respondent_id", how="left") \
                      .merge(demo_final, on="respondent_id", how="left")

print(f"Jumlah baris (responden): {len(feature_matrix)}")
print(f"Jumlah kolom fitur (di luar respondent_id): {feature_matrix.shape[1]-1}")
print(f"Rincian: base=7, actor_pivot=10, interaction=4, temporal=21, ecv=20, demo=8")

feature_matrix.to_csv("feature_matrix.csv", index=False)
print("\nSELESAI. feature_matrix.csv tersimpan.")

Jumlah baris (responden): 199
Jumlah kolom fitur (di luar respondent_id): 71
Rincian: base=7, actor_pivot=10, interaction=4, temporal=21, ecv=20, demo=8

SELESAI. feature_matrix.csv tersimpan.


---
## Ringkasan File 04

Feature matrix final: **199 baris × 70 kolom fitur** (respondent_id terpisah).

| Kelompok | Jumlah kolom |
|---|---|
| Skor dasar (VfV/FCNC/ID/MAS/ECI/GR/NREI keseluruhan) | 7 |
| Actor-specific pivot (VfV & NREI per aktor) | 10 |
| Interaction Features | 4 |
| Temporal Pattern Features | 21 |
| Ethical Consistency Vector | 20 |
| Demografi/legal | 8 |

**Catatan penting untuk File 05:** kolom `NREI_overall`, `MAS_overall`, `ECI_overall`, dan `NREI_{aktor}` (bagian B) **harus dikeluarkan** dari fitur (X) kalau target klasifikasinya adalah kategori NREI — supaya tidak data leakage (sudah kita bahas di awal). Silakan pilih salah satu:
- Target = kategori NREI → fitur yang aman dipakai: VfV/FCNC/ID (base + pivot + temporal + ECV) + interaction (kecuali yang melibatkan NREI) + demografi
- Kalau mau tetap pakai MAS/ECI sebagai fitur, target klasifikasinya jangan NREI (karena MAS & ECI menyumbang 80% ke NREI)